# Bunch fitting for the NC-CC mapping used for boosting purity of NC events
This script is used to extract the center positions of the CC events for MC events. The script is adapted from Yue's original fitting implementation, and follows from: https://github.com/S81D/BNB_bunches. We perform the fitting for truth-selected NC events (and CC) and then create a mapping between the two to inform our low-energy selection cuts.

In [ ]:
from lib.constants import *
from lib.imports import *
import warnings
from scipy.optimize import OptimizeWarning

# This cell is tailored to the Empirical model - we can instead use Base GENIE, but this script
# then requires some path editing

MC_directory = '../MC_GENIE-O16/'   # here for example
file_names = [file for file in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, file))]

POT = MC_POT_Empirical              # and here

plot_dir = 'plots/bunch_timing/MC/'
os.system('mkdir -p ' + plot_dir)


# Define NCQE and CC

def NCQE(clusterIndex,X_vertex,Y_vertex,Z_vertex,
         NC,QEL,PDG,MEC,Z,
         CH,bunch_time,CN,CEN_array,CN_array,BT_array,
        TankMRDCoinc,NoVeto,MRD_hitT,FMV_hitT,ClusterCharge,ChargeBalance):
    
    def reco_energy(pe):
        return (pe + 2.90)/5.90
    
    bunch_time_cutoff = 1560
    time_to_prompt_end = 258
    
    # 1. true NCQE selection within FV
    if np.sqrt((X_vertex)**2 + (Z_vertex)**2) > FV_radius or np.abs(Y_vertex) > FV_half_height:
        return False
    
    if NC == 0:
        return False
    if QEL == 0:
        return False
    if Z != 8:
        return False
    if PDG == -14:
        return False
    if PDG == -12:
        return False
    
    # 2. Minimum 10 hits
    if CH < 10:
        return False
        
    # 3. Cluster time within 1.6 us beam spill
    if bunch_time > bunch_time_cutoff:
        return False
    
    # 4. Only one prompt cluster
    if CN != 0:
        return False
    skip_event = False
    i = clusterIndex
    for k in range(i+1,len(CN_array)):        # look at the subsequent clusters in this event
        if CN_array[k] != 0:   # if the next cluster is a secondary cluster
            # check if the time of the next cluster is within the prompt 2us window
            if BT_array[k] < bunch_time_cutoff + time_to_prompt_end:
                skip_event = True
                break
        else:     # we found an event without a secondary cluster
            break
    if skip_event:
        return False  # Skip the entire event
        
    # 5. No FMV or MRD activity
    if TankMRDCoinc == 1 or NoVeto == 0 or len(MRD_hitT) != 0 or len(FMV_hitT) != 0:
        return False
        
    # 7. basic CCB cut
    if ChargeBalance >= 0.6:
        return False
    
    # 8. reco E
    recoE = reco_energy(ClusterCharge)
    if recoE < 5 or recoE > 12:
        return False
    
    # if the event passes
    return True
    


def CCinc(clusterIndex,TankMRDCoinc,Track,NoVeto,FMV_hitT,ClusterNumber,
          bunch_time,ClusterCharge,CN_array,CPE_array,ChargeBalance,ClusterHits):
    
    bunch_time_cutoff = 1560
        
    # 2. No Veto activity
    if NoVeto == 0 or len(FMV_hitT) != 0:
        return False
        
    # 3. Prompt cluster (and within the 1.6 us beam spill)
    if ClusterNumber != 0:
        return False
    if bunch_time > bunch_time_cutoff:
        return False
    
    # 4. Charge balance < 0.2 and 4000 > charge > 1000
    if ChargeBalance > 0.2 or ChargeBalance < 0:
        return False
        
    # 5. Brightest cluster in the event
    brightest = ClusterCharge; flag = False
    i = clusterIndex
    for j in range(i+1, len(CN_array)):
        if CN_array[j] == 0:
            break
        if CPE_array[j] > brightest:
            flag = True
            break
    if flag == True:
        return False
    
    # 6. Must have 100 PMT hits
    if ClusterHits < 100:
        return False
    
    # if it passes
    return True


print('done')

In [ ]:
# Load in MC files (this will take a while since we're doing both CC and NC selection

bunch_times_CCinc = [];
bunch_times_NCQE = []; 
bunch_times_Ext = [];           # dirt events (just to investigate if needed)

for file in trange(len(file_names)):
    
    root = uproot.open(MC_directory + file_names[file])

    # cluster info
    T = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array()
    CPE = T['clusterPE'].array()
    CCB = T['clusterChargeBalance'].array()
    CH = T['clusterHits'].array()
    BT = T['bunchTimes'].array()
    TMC = T['TankMRDCoinc'].array()
    NV1 = T['NoVeto'].array()
    CN = T['clusterNumber'].array()

    # trigger info
    S = root['phaseIITriggerTree']
    TEN = S['eventNumber'].array()
    MRDT = S['numMRDTracks'].array()
    veto_t = S['FMVhitT'].array()
    vx = S['trueNuIntxVtx_X'].array()
    vy = S['trueNuIntxVtx_Y'].array()
    vz = S['trueNuIntxVtx_Z'].array()
    nc = S['trueNC'].array()
    qe = S['trueQEL'].array()
    pd = S['trueFSLPdg'].array()
    mec = S['trueMEC'].array()
    z = S['trueTargetZ'].array()
    mrd_t = S['MRDhitT'].array()
    veto_t = S['FMVhitT'].array()


    for i in range(len(CEN)):

        j = CEN[i]
        
        vertex_X = vx[j]/100; vertex_Y = vy[j]/100 + 0.1446; vertex_Z = vz[j]/100 - 1.681

        # ------------------------------------------------ #
        # CCinc events
        CC_muon = CCinc(i,TMC[i],MRDT[j],NV1[i],veto_t[j],CN[i],BT[i],CPE[i],CN,CPE,CCB[i],CH[i])
        if CC_muon == True:
            if np.sqrt(vertex_X**2 + vertex_Z**2) > radius or np.abs(vertex_Y) > half_height:
                bunch_times_Ext.append(BT[i])
            else:
                bunch_times_CCinc.append(BT[i])

        # ------------------------------------------------ #
        # Now for the true NCQE events within our FV
        ncqe_event = NCQE(i,vertex_X,vertex_Y,vertex_Z,nc[j],qe[j],pd[j],mec[j],z[j],
                          CH[i],BT[i],CN[i],CEN,CN,BT,TMC[i],NV1[i],mrd_t[j],veto_t[j],CPE[i],CCB[i])
        if ncqe_event == True:
            bunch_times_NCQE.append(BT[i])    

        
print('POT:',POT)
print('NCQE:',len(bunch_times_NCQE))
print('CCinc:',len(bunch_times_CCinc))
print('\ndone')

In [ ]:
# scale down the MC external events
# Recall that the dirt events have a different (and sloppier) timing structure than the neutrino interactions occuring within the tank
# we therefore must down-scale the dirt events to ensure the bunch structure is accurate to data

rng = np.random.default_rng(seed=42)    # optional seed for reproducibility

# Convert to array if not already
ext = np.asarray(bunch_times_Ext)

# Number of events to keep
n_keep = int(ALPHA_SCALING * len(ext))           # scaling from CC sideband analysis: Calculated Scale Factor: 0.2697

# Randomly sample without replacement
ext_sampled = rng.choice(ext, size=n_keep, replace=False)

# Combine with CCinc
bunch_times_CCinc_ext = np.concatenate([bunch_times_CCinc, ext_sampled])

print('Events scaled down by x' + str(ALPHA_SCALING))

In [ ]:
# Fitting machinery

def gaussian(x, amp, mean, sigma):
    """Gaussian function for fitting"""
    return amp * np.exp(-((x - mean) ** 2) / (2 * sigma ** 2))

def fit_gaussian_constrained(x, y, initial_mean, fixed_spacing, bunch_idx):
    """
    Fit gaussian with mean constrained near expected position.
    More lenient bounds for low statistics.
    """
    if len(x) < 3 or np.sum(y) < 5:  # Skip if too few points or events
        return None, None
    
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=OptimizeWarning)
            
            # More generous bounds for low stats
            mean_tolerance = 8  # Allow ±8 ns deviation
            amp_guess = max(y) if max(y) > 0 else 1
            sigma_guess = 2.0  # Start closer to expected
            
            params, pcov = curve_fit(
                gaussian, x, y,
                p0=[amp_guess, initial_mean, sigma_guess],
                bounds=(
                    [0, initial_mean - mean_tolerance, 0.5],  # Lower bounds
                    [amp_guess * 3, initial_mean + mean_tolerance, 10]  # Upper bounds
                ),
                maxfev=10000
            )
            perr = np.sqrt(np.diag(pcov))
            return params, perr
    except (RuntimeError, ValueError) as e:
        return None, None

def compute_fit_quality(x, y, params):
    """
    Compute fit quality metric that works better for low statistics.
    Uses weighted residuals instead of chi-square.
    """
    if params is None:
        return np.inf
    
    y_fit = gaussian(x, *params)
    
    # For low statistics, use weighted sum of squared residuals
    # Add small constant to avoid division by zero
    weights = 1.0 / (np.sqrt(y + 1))  # Poisson-like weighting
    residuals = (y - y_fit) * weights
    
    return np.sum(residuals**2)

def perform_stable_fit(timing_data, id_name, fixed_spacing, first_peak_guess, 
                       SavePath, n_bunches=81, scan_range=20):
    """
    Perform bunch fitting with FIXED spacing, only scanning global offset.
    
    Parameters:
    -----------
    timing_data : array
        Event times
    id_name : str
        Identifier for plots
    fixed_spacing : float
        Known bunch spacing (18.936 ns)
    first_peak_guess : float
        Initial guess for first bunch center
    SavePath : str
        Directory for saving plots
    n_bunches : int
        Number of bunches (default 81)
    scan_range : float
        Range to scan around first_peak_guess (±scan_range ns)
    """

    # ***************************************************** #
    # modify to filter out baby bunch or unwanted fit regions
    #bins_TSDt = np.arange(0, 1600, 2)
    bins_TSDt = np.arange(0, 1555, 2)
                         
    h_TSDt, _ = np.histogram(timing_data, bins=bins_TSDt)
    bin_centers = (bins_TSDt[:-1] + bins_TSDt[1:]) / 2
    
    # Scan only the global offset
    scan_step = 0.2  # Coarser steps since spacing is fixed
    scan_offsets = np.arange(first_peak_guess - scan_range, 
                             first_peak_guess + scan_range, 
                             scan_step)
    
    best_quality = np.inf
    best_offset = first_peak_guess
    best_results = None
    
    print(f"Scanning {len(scan_offsets)} offset positions with fixed spacing={fixed_spacing} ns")
    
    for offset in trange(len(scan_offsets), desc="Finding best global offset"):
        current_offset = scan_offsets[offset]
        
        bunch_means = []
        bunch_sigmas = []
        bunch_mean_errors = []
        fit_qualities = []
        
        # Fit all bunches with this offset
        for i in range(n_bunches):
            bunch_center = current_offset + i * fixed_spacing
            
            # Define window around expected bunch position
            half_window = fixed_spacing * 0.45  # Use 90% of spacing
            start = bunch_center - half_window
            end = bunch_center + half_window
            
            mask = (bin_centers >= start) & (bin_centers < end)
            x = bin_centers[mask]
            y = h_TSDt[mask]
            
            if len(x) < 3:
                bunch_means.append(np.nan)
                bunch_sigmas.append(np.nan)
                bunch_mean_errors.append(np.nan)
                fit_qualities.append(np.inf)
                continue
            
            params, errors = fit_gaussian_constrained(
                x, y, bunch_center, fixed_spacing, i
            )
            
            if params is None:
                bunch_means.append(np.nan)
                bunch_sigmas.append(np.nan)
                bunch_mean_errors.append(np.nan)
                fit_qualities.append(np.inf)
            else:
                bunch_means.append(params[1])
                bunch_sigmas.append(params[2])
                bunch_mean_errors.append(errors[1])
                fit_qualities.append(compute_fit_quality(x, y, params))
        
        # Total quality for this offset (sum of all bunch fit qualities)
        total_quality = np.nansum(fit_qualities)
        
        if total_quality < best_quality:
            best_quality = total_quality
            best_offset = current_offset
            best_results = {
                'offset': current_offset,
                'means': bunch_means,
                'sigmas': bunch_sigmas,
                'mean_errors': bunch_mean_errors,
                'qualities': fit_qualities
            }
    
    print(f"\nBest global offset: {best_offset:.2f} ns")
    print(f"Best fit quality score: {best_quality:.2f}")
    
    # Filter out failed fits
    valid_means = [m for m in best_results['means'] if not np.isnan(m)]
    valid_sigmas = [s for s in best_results['sigmas'] if not np.isnan(s)]
    
    print(f"Successfully fitted {len(valid_means)}/{n_bunches} bunches")
    print(f"Mean bunch width: {np.mean(valid_sigmas):.3f} ± {np.std(valid_sigmas):.3f} ns")
    
    # Generate plots
    plot_timing_histogram(bins_TSDt, h_TSDt, id_name, SavePath)
    plot_fitted_bunches(bins_TSDt, h_TSDt, best_results, fixed_spacing, 
                        id_name, SavePath)
    plot_bunch_diagnostics(best_results, fixed_spacing, id_name, SavePath)
    
    return best_results

def plot_timing_histogram(bins_TSDt, h_TSDt, id_name, SavePath):
    """Plot basic timing histogram"""
    plt.figure(figsize=(12, 4))
    plt.hist(bins_TSDt[:-1], bins=bins_TSDt, weights=h_TSDt, 
             alpha=0.7, color='tomato', histtype='stepfilled')
    plt.hist(bins_TSDt[:-1], bins=bins_TSDt, weights=h_TSDt, 
             alpha=0.7, color='black', histtype='step', linewidth=2)
    plt.xlabel('cluster time [ns]', fontsize=16, loc='right')
    plt.ylabel('events / 2ns', fontsize=16)
    plt.title(f'Beam Timing Histogram for {id_name}', fontsize=24)
    ax = plt.gca()
    ax.tick_params(axis='both', direction='in')
    ax.xaxis.set_major_locator(MultipleLocator(250))
    ax.xaxis.set_minor_locator(MultipleLocator(50))
    #plt.text(0.2, 0.93, f'{total_POT_TOR860:.2f}' + r'$\times 10^{20}$' + ' POT', 
    #     horizontalalignment='right', verticalalignment='top', 
    #     transform=plt.gca().transAxes, fontsize=14)
    plt.tight_layout()
    plt.savefig(SavePath + f'TimingHist_{id_name}.png', dpi=300, 
                bbox_inches='tight', pad_inches=.3, facecolor='w')
    plt.close()

def plot_fitted_bunches(bins_TSDt, h_TSDt, results, fixed_spacing, 
                        id_name, SavePath):
    """Plot histogram with fitted Gaussians overlaid"""
    bin_centers = (bins_TSDt[:-1] + bins_TSDt[1:]) / 2
    
    plt.figure(figsize=(12, 4))
    plt.hist(bins_TSDt[:-1], bins=bins_TSDt, weights=h_TSDt, 
             alpha=0.7, color='C0')
    
    for i, mean in enumerate(results['means']):
        if np.isnan(mean):
            continue
        
        sigma = results['sigmas'][i]
        half_window = fixed_spacing * 0.45
        start = mean - half_window
        end = mean + half_window
        
        mask = (bin_centers >= start) & (bin_centers < end)
        x = bin_centers[mask]
        y = h_TSDt[mask]
        
        if len(x) < 3:
            continue
        
        # Refit for plotting
        params, _ = fit_gaussian_constrained(x, y, mean, fixed_spacing, i)
        if params is not None:
            x_smooth = np.linspace(start, end, 100)
            y_fit = gaussian(x_smooth, *params)
            plt.plot(x_smooth, y_fit, 'r-', linewidth=1, alpha=0.8)
    
    plt.xlabel('cluster time [ns]', loc='right', fontsize=16)
    plt.ylabel('events / 2ns', fontsize=16)
    plt.title(f'Beam Timing with Gaussian Fits for {id_name}', fontsize=24)
    plt.tight_layout()
    plt.savefig(SavePath + f'TimingHist_gaussian_fitting_{id_name}.png', 
                dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
    plt.close()

def plot_bunch_diagnostics(results, fixed_spacing, id_name, SavePath):
    """Plot diagnostic plots for bunch fitting"""
    valid_means = [m for m in results['means'] if not np.isnan(m)]
    valid_sigmas = [s for s in results['sigmas'] if not np.isnan(s)]
    
    # Bunch spacing intervals
    intervals = np.diff([m for m in results['means'] if not np.isnan(m)])
    
    if len(intervals) > 0:
        plt.figure(figsize=(5, 4))
        plt.hist(intervals, range=(15, 25), bins=20, alpha=0.7, color='C0')
        plt.axvline(x=fixed_spacing, color='r', linestyle='--', linewidth=2,
                   label=f'Expected = {fixed_spacing} ns')
        plt.xlabel('Fitted Interval [ns]', loc='right')
        plt.title(f'Bunch spacings for {id_name}')
        plt.legend(frameon=False, fontsize = 12)
        plt.tight_layout()
        plt.savefig(SavePath + f'bunch_spacings_{id_name}.png', dpi=300,
                   bbox_inches='tight', pad_inches=.3, facecolor='w')
        plt.close()
    
    # Bunch widths
    if len(valid_sigmas) > 0:
        sigma_mean = np.mean(valid_sigmas)
        plt.figure(figsize=(5, 4))
        plt.hist(valid_sigmas, bins=20, alpha=0.7, color='C0', range=(0, 9))
        plt.axvline(x=sigma_mean, color='r', linestyle='--', linewidth=2,
                   label=f'σ mean = {sigma_mean:.2f} ns')
        plt.xlabel(r'$\sigma$ [ns]', loc='right')
        plt.title(f'Bunch widths for {id_name}')
        plt.legend(frameon=False, fontsize = 12)
        plt.tight_layout()
        plt.savefig(SavePath + f'bunch_sigma_{id_name}.png', dpi=300,
                   bbox_inches='tight', pad_inches=.3, facecolor='w')
        plt.close()
        
def superimpose_bunches(timing_data, results, fixed_spacing, id_name, SavePath,
                        fit_range_fraction=0.67):
    """
    Superimpose all bunches to measure intrinsic bunch width.
    
    Parameters:
    -----------
    timing_data : array
        Original event times
    results : dict
        Output from perform_stable_fit
    fixed_spacing : float
        Bunch spacing (18.936 ns)
    id_name : str
        Identifier for plots
    SavePath : str
        Directory for saving
    fit_range_fraction : float
        Fraction of half-spacing to use for Gaussian fit (default 0.8)
    """
    
    centered_times = []
    valid_bunch_count = 0
    
    # Shift each bunch to center at zero
    for bunch_idx, bunch_center in enumerate(results['means']):
        if np.isnan(bunch_center):
            continue
        
        # Define window around this bunch
        half_width = fixed_spacing / 2
        timings = np.array(timing_data)
        mask = (timings > bunch_center - half_width) & (timings < bunch_center + half_width)
        
        filtered_times = timings[mask]
        
        if len(filtered_times) < 2:  # Skip empty bunches
            continue
        
        # Shift to center at zero
        shifted_times = filtered_times - bunch_center
        centered_times.extend(shifted_times)
        valid_bunch_count += 1
    
    if len(centered_times) == 0:
        print("Warning: No events found for superposition!")
        return None
    
    print(f"\nSuperimposed {valid_bunch_count} bunches with {len(centered_times)} total events")
    
    # Create histogram
    half_range = fixed_spacing / 2
    ranges = (-np.floor(half_range), np.ceil(half_range))
    n_bins = int(ranges[1] - ranges[0])  # 1 ns bins
    
    plt.figure(figsize=(5, 4))
    counts, bin_edges, _ = plt.hist(centered_times, bins=n_bins, range=ranges, 
                                     alpha=1, color='navy', histtype='step', linewidth=2)
    
    # Prepare for Gaussian fit
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Fit range: use central fraction to avoid tails from adjacent bunches
    fit_half_range = half_range * fit_range_fraction
    mask = (bin_centers >= -fit_half_range) & (bin_centers <= fit_half_range)
    filtered_bin_centers = bin_centers[mask]
    filtered_counts = counts[mask]
    
    # Fit Gaussian to superimposed distribution
    try:
        initial_guess = [max(filtered_counts), 0.0, 2]  # Expect ~2 ns width
        popt, pcov = curve_fit(gaussian, filtered_bin_centers, filtered_counts, 
                               p0=initial_guess,
                               bounds=([0, -2, 0.5], [np.inf, 2, 6]))
        
        amplitude, mean, sigma = popt
        sigma_error = np.sqrt(pcov[2, 2])
        
        # Plot fit
        x_fit = np.linspace(-fit_half_range, fit_half_range, 1000)
        y_fit = gaussian(x_fit, amplitude, mean, sigma)
        plt.plot(x_fit, y_fit, 'r-', linewidth=2.5, label='Gaussian fit')
        
        # Add text annotations
        #plt.text(0.97, 0.95, f'{total_POT_TOR860:.2f}' + r'$\times 10^{20}$' + ' POT', 
        #     horizontalalignment='right', verticalalignment='top', 
        #     transform=plt.gca().transAxes, fontsize=12)
        plt.text(0.97, 0.95, r'$\text{N}_{\text{events}} = $' + str(len(centered_times)), 
                horizontalalignment='right', verticalalignment='top', 
                transform=plt.gca().transAxes, fontsize=12)
        plt.text(0.97, 0.85, f'$\\sigma = {sigma:.2f} \\pm {sigma_error:.2f}$ ns', 
                horizontalalignment='right', verticalalignment='top', 
                transform=plt.gca().transAxes, fontsize=12)
        
        print(f"Superimposed bunch fit:")
        print(f"  Mean: {mean:.4f} ns")
        print(f"  Sigma: {sigma:.4f} ± {sigma_error:.4f} ns")
        print(f"  (Expected BNB sigma ~ 1.308 ns)")
        
    except RuntimeError as e:
        print(f"Failed to fit superimposed distribution: {e}")
        sigma, sigma_error, mean = np.nan, np.nan, np.nan
    
    # Formatting
    plt.xlabel('Time relative to bunch center [ns]', fontsize=16)
    plt.ylabel('Events / ns', fontsize=16)
    plt.title(f'{id_name}', fontsize=20)
    
    ax = plt.gca()
    ax.tick_params(axis='both', direction='in')
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.xaxis.set_minor_locator(MultipleLocator(1))
    
    plt.xlim([ranges[0], ranges[1]])
    plt.tight_layout()
    plt.savefig(SavePath + f'superimposed_bunches_{id_name}.png', 
                dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
    plt.show()
    
    return {
        'mean': mean,
        'sigma': sigma,
        'sigma_error': sigma_error,
        'n_events': len(centered_times),
        'n_bunches': valid_bunch_count
    }


def linear_func(x, a, b):   # for fitting bunch spacing
    return a * x + b

# you will get back:
# - linear fit to spacing
# - bunch sigma distribution
# - bunch spacing distribution
# - superimposed bunch distribution
# - beam timing plots (w/ and w/out fitted gaussians)

print('done')

In [ ]:
# intialization --> we will do two sequential fitting runs

id_name_NC = "MC NCQE"
id_name_CC = "MC CC"
array_name_NC = bunch_times_NCQE
array_name_CC = bunch_times_CCinc_ext        # includes the down-sampled dirt events

#  '''                                              '''
# ---------------------------------------------------- #
#  '''                                              '''

# First pass will be NC
print('NC ' + '_'*40)

# Step 1: Fit bunches with fixed spacing (adjust guesses accordingly)
results = perform_stable_fit(
    timing_data=array_name_NC,
    id_name=id_name_NC,
    fixed_spacing=18.936,
    first_peak_guess=28,        # decent guess for MC samples w/ AssignBunchTiming tool
    SavePath=plot_dir,
    n_bunches=81,
    scan_range=20
)

# Step 2: Linear fit to bunch centers (to measure actual spacing)
valid_indices = [i for i, m in enumerate(results['means']) if not np.isnan(m)]
x_data = np.array(valid_indices)
y_data = np.array([results['means'][i] for i in valid_indices])

params, covariance = curve_fit(linear_func, x_data, y_data)
slope = params[0]
intercept = params[1]
slope_error = np.sqrt(covariance[0][0])

print(f"Measured bunch spacing: {slope:.4f} ± {slope_error:.4f} ns")
print(f"Expected spacing: 18.936 ns")
print(f"Difference: {slope - 18.936:.4f} ns")

# Plot linear fit
plt.figure(figsize=(5, 4))
plt.scatter(x_data, y_data, label='Fitted bunch centers', s=10)
plt.plot(x_data, linear_func(x_data, *params), 'r-', 
         label=f'Fit: {slope:.4f} ± {slope_error:.4f} ns')
plt.xlabel('Bunch number')
plt.ylabel('Bunch Center time [ns]')
plt.title('Linear fit to bunch spacings')
plt.legend(fontsize=12)
ax = plt.gca()
ax.tick_params(axis='both', direction='in')
ax.xaxis.set_major_locator(MultipleLocator(20))
ax.xaxis.set_minor_locator(MultipleLocator(5))
plt.tight_layout()
plt.savefig(plot_dir + f'bunch_linear_fit_spacings_{id_name_NC}.png', 
            dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
plt.close()

# Step 3: Superimpose bunches
superimposed = superimpose_bunches(
    timing_data=array_name_NC,
    results=results,
    fixed_spacing=slope,  # Use measured spacing, not 18.936
    id_name=id_name_NC,
    SavePath=plot_dir,
    fit_range_fraction=0.6  # Fit central 80% to avoid tails
)

# record the bunch centers
bunch_centers_NC = results['means']              # List of fitted bunch positions
bunch_widths_NC = results['sigmas']              # List of fitted widths
bunch_center_errors_NC = results['mean_errors']  # List of errors on bunch positions



# Second pass is CC
print('CC ' + '_'*40)

# Step 1: Fit bunches with fixed spacing (adjust guesses accordingly)
results = perform_stable_fit(
    timing_data=array_name_CC,
    id_name=id_name_CC,
    fixed_spacing=18.936,
    first_peak_guess=28,        # decent guess for MC samples w/ AssignBunchTiming tool
    SavePath=plot_dir,
    n_bunches=81,
    scan_range=20
)

# Step 2: Linear fit to bunch centers (to measure actual spacing)
valid_indices = [i for i, m in enumerate(results['means']) if not np.isnan(m)]
x_data = np.array(valid_indices)
y_data = np.array([results['means'][i] for i in valid_indices])

params, covariance = curve_fit(linear_func, x_data, y_data)
slope = params[0]
intercept = params[1]
slope_error = np.sqrt(covariance[0][0])

print(f"Measured bunch spacing: {slope:.4f} ± {slope_error:.4f} ns")
print(f"Expected spacing: 18.936 ns")
print(f"Difference: {slope - 18.936:.4f} ns")

# Plot linear fit
plt.figure(figsize=(5, 4))
plt.scatter(x_data, y_data, label='Fitted bunch centers', s=10)
plt.plot(x_data, linear_func(x_data, *params), 'r-', 
         label=f'Fit: {slope:.4f} ± {slope_error:.4f} ns')
plt.xlabel('Bunch number')
plt.ylabel('Bunch Center time [ns]')
plt.title('Linear fit to bunch spacings')
plt.legend(fontsize=12)
ax = plt.gca()
ax.tick_params(axis='both', direction='in')
ax.xaxis.set_major_locator(MultipleLocator(20))
ax.xaxis.set_minor_locator(MultipleLocator(5))
plt.tight_layout()
plt.savefig(plot_dir + f'bunch_linear_fit_spacings_{id_name_CC}.png', 
            dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
plt.close()

# Step 3: Superimpose bunches
superimposed = superimpose_bunches(
    timing_data=array_name_CC,
    results=results,
    fixed_spacing=slope,  # Use measured spacing, not 18.936
    id_name=id_name_CC,
    SavePath=plot_dir,
    fit_range_fraction=0.6  # Fit central 80% to avoid tails
)

# record the bunch centers
bunch_centers_CC = results['means']              # List of fitted bunch positions
bunch_widths_CC = results['sigmas']              # List of fitted widths
bunch_center_errors_CC = results['mean_errors']  # List of errors on bunch positions

# -----------------------------------------------------


print('\ndone!\n')

In [ ]:
# Now you can extract the mapping from NC:CC

diff = []; diff_std = []
for i in range(len(bunch_center_errors_NC)):
    diff.append(bunch_centers_CC[i] - bunch_centers_NC[i])
    diff_std.append( np.sqrt( bunch_center_errors_CC[i]**2 + bunch_center_errors_NC[i]**2 ) )
    
diff = np.array(diff)
diff_std = np.array(diff_std)

# Weighted average and error
weights = 1.0 / (diff_std**2)
weighted_mean = np.sum(weights * diff) / np.sum(weights)
weighted_mean_err = np.sqrt(1.0 / np.sum(weights))

print(f"Weighted average difference = {weighted_mean:.4f} ± {weighted_mean_err:.4f} ns")
    
x_points = [i for i in range(len(diff))]

fig, ax = plt.subplots(figsize=(5, 3))
plt.errorbar(x_points, diff, yerr=diff_std, fmt='o', color='k', label='MC bunch time difference')
plt.axhline(weighted_mean, color = 'royalblue', label = f"{weighted_mean:.3f} ± {weighted_mean_err:.3f} ns")
plt.axhspan(weighted_mean - weighted_mean_err, 
            weighted_mean + weighted_mean_err, 
            color='royalblue', alpha=0.2)
plt.xlabel('Bunch number')
plt.ylabel(r'$\Delta t(\text{CC} - \text{NCQE})$')
plt.ylim([-2.5,5]); plt.xlim([-1, 82])
ax.xaxis.set_minor_locator(MultipleLocator(2))
plt.legend(fontsize = 12, frameon = False)
plt.tight_layout()
plt.savefig(f'{plot_dir}/bunch_time_difference_NCQE_CCinc_scale_{ALPHA_SCALING}.png',
            dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()

print('\ndone')

In [ ]:
# Access CC center fits and their uncertainties:
print('Centers\n', [float(x) for x in bunch_centers_CC], '\n\n')
print('Errors\n', [float(x) for x in bunch_center_errors_CC], '\n')